In [1]:
# Import necessary libraries

import asyncio
import io
import os
from tkinter import Tk, filedialog
import edge_tts
import pandas as pd
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.11.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# # Run this to inspect one single raw voice entry dictionary
# v_manager = await edge_tts.VoicesManager.create()
# print(v_manager.voices[0].keys())
# print(v_manager.voices[0])

In [9]:
# Fetch and display the list of available voices
async def get_comprehensive_voice_table():
    voices_manager = await edge_tts.VoicesManager.create()
    voices = voices_manager.voices
    
    processed_voices = []
    for v in voices:
        # Safely extract the nested VoiceTag data if it exists
        voice_tag = v.get("VoiceTag", {})
        categories = voice_tag.get("ContentCategories", [])
        personalities = voice_tag.get("VoicePersonalities", [])
        
        processed_voices.append({
            "ShortName": v.get("ShortName"),
            "Gender": v.get("Gender"),
            "Personalities": ", ".join(personalities) if personalities else "None",
            "Locale": v.get("Locale"),
            "Language": v.get("Language"),
            "FriendlyName": v.get("FriendlyName"),
            "SuggestedCodec": v.get("SuggestedCodec"),
            "Categories": ", ".join(categories) if categories else "None"
        })
    
    return pd.DataFrame(processed_voices)

# Generate the master DataFrame table variable
voice_df = await get_comprehensive_voice_table()

# Display the first 5 rows of the interactive table in Jupyter
voice_df.head(5)

# To see all rows, uncomment the line below
# voice_df

# Filter the table to only include rows where the Language is English
english_voices_df = voice_df[voice_df['Language'] == 'en']


In [4]:
# Define the text-to-speech function that uses edge-tts to convert text to 
#   speech and returns an in-memory audio buffer

async def text_to_speech(text, rate="0%", voice="en-US-AnaNeural"):
    """Streams TTS audio into an in-memory buffer."""
    communicate = edge_tts.Communicate(text=text, rate=rate, voice=voice)
    audio_buffer = io.BytesIO()
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_buffer.write(chunk["data"])
    audio_buffer.seek(0)
    return audio_buffer

In [5]:
# Define a function to prompt the user to select a text file

def prompt_for_text_file():
    """Opens a native Windows file explorer to select a text file."""
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True) # Forces window to the front
    print("Opening file selector... Please choose a text file.")
    file_path = filedialog.askopenfilename(
        title="Select a Text File to Read Aloud",
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
    )
    root.destroy()
    return file_path

In [6]:
# Define the main function to read the selected text file aloud using the TTS function

async def read_text_file_aloud(file_path, rate="-7%", voice="en-US-JennyNeural"):
    """Reads content from the selected text file and plays it chunk by chunk."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]
    except Exception as e:
        print(f"Error reading file: {e}")
        return

    if not lines:
        print("The selected file is empty.")
        return

    print(f"\nPlaying: {os.path.basename(file_path)}")
    print(f"Voice: {voice} | Speed: {rate}\n")
    
    pygame.mixer.init()
    try:
        for chunk in lines:
            print(f"Reading: {chunk[:50]}...")
            audio_buffer = await text_to_speech(chunk, rate, voice)
            pygame.mixer.music.load(io.BytesIO(audio_buffer.getvalue()))
            pygame.mixer.music.play()
            while pygame.mixer.music.get_busy():
                await asyncio.sleep(0.2)
            await asyncio.sleep(0.4) 
    except Exception as e:
        print(f"Error during playback: {e}")
    finally:
        pygame.mixer.quit()

In [18]:
# Select a text file to be loaded

selected_file = prompt_for_text_file()

if selected_file:
    print(f"Successfully loaded: {os.path.basename(selected_file)}")
else:
    print("No file selected.")

Opening file selector... Please choose a text file.
Successfully loaded: A small bird.txt


In [19]:
# This cell is for testing the TTS functionality with the selected file. 
# You can adjust the voice and speed settings as needed.
# Uncomment the 
# Note: Make sure to run this cell after selecting a file in the previous cell.

# Quick settings adjustments

voice_choice = "en-AU-NatashaNeural"
speed_rate = "-5%"

if 'selected_file' in locals() and selected_file:
    # Notice the direct 'await' keyword used here instead of asyncio.run()
    await read_text_file_aloud(selected_file, rate=speed_rate, voice=voice_choice)
else:
    print("Please select a file in the previous cell first!")


Playing: A small bird.txt
Voice: en-AU-NatashaNeural | Speed: -5%

Reading: A small bird flew through the forest, searching fo...
